# Set FRP markers based on special fixations

In [ ]:
from tqdm import tqdm

from main.frp_utils.eeg_helpers import *
from main.frp_utils.file_helpers import *
from main.frp_utils.path_helpers import *
from main.frp_utils.textconstants import *

In [ ]:
# get all files from participants
folders = get_participant_folder_per_participant()
print(folders)

## Calculate fixation for FRP and set markers

In [ ]:
for participant in tqdm(folders):
    print('----------------------------------------------')
    print(PARTICIPANT, participant)
    # skip participant if excluded
    exclusions = get_exclusions(
        participant, [PARTICIPANT], DataModes)[PARTICIPANT]
    if any(exclusions.values()):
        print('excluded')
        continue

    exclusions = get_exclusions(participant, [SNIPPET], DataModes)[SNIPPET]
    snippets = [snippet for snippet in exclusions if not any(exclusions[snippet].values())]

    print('(01/04) Load special fixation data')
    special_fixation_data = pd.read_csv(get_special_fixations_path(participant), index_col=False, sep=';')

    print('(02/04) Load EEG data')
    eeg_trials = get_all_eeg_trial_segments(True, participant)

    print('(03/04) Load Behavioral data')
    behavioral_data = pd.read_csv(get_behavioral_data_path(
        'total', final_data_exclusion=True), index_col=False, sep=';', dtype={PARTICIPANT: str})
    behavioral_data = behavioral_data[behavioral_data[PARTICIPANT] == participant]

    print('(04/04) Calculate and add FRP positions and export FRP trial')
    add_frp_marker_by_special_fixations(participant, snippets, behavioral_data, special_fixation_data, eeg_trials)